## Setup

Combines the two neighborhood-analysis notebooks into one per-sample loop.

**Removed:** co-occurrence analysis (`sq.gr.co_occurrence`, ~2hr/sample) and all of the custom blue/red recoloring of the enrichment matrix — only the default `RdBu_r` matrix (annotated + not) is kept.

In [1]:
import os
import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


## Parameters

Same `sample_nm_list` as the scoring notebook. All outputs (figures + the neighbor-graph `.h5ad`) go to the same shared `odir` used there.

In [6]:
#sample_nm_list = ['HL170058_filt', 'HL230324_filt', 'HL160029_filt']
sample_nm_list = ['HL170058_filt']

odir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/outputs/sandbox/RNA/clusterprofiler_kegg_react/top_deseq2_genes_all_samples/'
os.makedirs(odir, exist_ok=True)


## Shared config (sample-independent)

In [3]:
cluster_key = 'celltype_proj_heps_pt02_cp'

# Row order used when pulling out just the hepatocyte subtypes from the
# full enrichment matrix (Senescent.High listed first)
heps_labels = [
    'Hepatocytes.Senescent.High', 'Hepatocytes.Senescent', 'Hepatocytes.Zone1',
    'Hepatocytes.Zone2', 'Hepatocytes.Zone3',
]


## Per-sample pipeline

For each sample: load the full tangram-predicted spatial object plus the hepatocyte-subtype-labeled object from the scoring notebook, map the subtype labels back onto the full object, run `spatial_neighbors` + `nhood_enrichment` (no co-occurrence), save the full enrichment matrix, write the neighbor-graph `.h5ad`, then save a zoomed-in heatmap of just the hepatocyte subtypes.

In [13]:
def process_sample(sample_nm):
    """Run the neighborhood-enrichment pipeline (no co-occurrence) for one sample.

    Loads the tangram-predicted spatial object and the hepatocyte-subtype-labeled
    object (from the scoring notebook), maps subtype labels onto the full object,
    computes neighborhood enrichment, saves the enrichment-matrix figures, writes
    the neighbor-graph .h5ad, and saves the zoomed-in hepatocyte-subtype heatmap.

    Returns the final AnnData object (adata_sub) for this sample.
    """
    # --- load inputs ---
    adata_path = (
        f"/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/"
        f"misc/tangram_scores_dist2/{sample_nm}/adata_0.5_tangram_filt.h5ad"
    )
    adata = sc.read_h5ad(adata_path)

    adata_heps_path = f"{odir}adata_heps_score_sencells_labels_{sample_nm}.h5ad"
    adata_heps = sc.read_h5ad(adata_heps_path)

    full_adata = adata.copy()

    # ================= TROUBLESHOOTING: pre-merge diagnostics =================
    print(f"[{sample_nm}] --- troubleshooting celltype_proj_heps_pt02 merge ---")
    print(f"[{sample_nm}] pred_cell_types value counts (full_adata, before merge):")
    print(full_adata.obs['pred_cell_types'].value_counts())

    heps_idx = set(adata_heps.obs_names)
    full_idx = set(full_adata.obs_names)
    n_dup_heps = adata_heps.obs_names.duplicated().sum()
    n_dup_full = full_adata.obs_names.duplicated().sum()
    print(f"[{sample_nm}] adata_heps n_obs: {len(heps_idx)} | full_adata n_obs: {len(full_idx)}")
    print(f"[{sample_nm}] barcode overlap (adata_heps ∩ full_adata): {len(heps_idx & full_idx)}")
    print(f"[{sample_nm}] duplicate barcodes -> adata_heps: {n_dup_heps} | full_adata: {n_dup_full}")
    if len(heps_idx & full_idx) < len(heps_idx):
        print(
            f"[{sample_nm}] WARNING: {len(heps_idx - full_idx)} adata_heps barcodes "
            f"have no match in full_adata -- index/barcode mismatch likely."
        )
    # ============================================================================

    # --- map the hepatocyte subtype labels (pt02) back onto the full spatial object ---
    hepatocyte_mapping = adata_heps.obs['celltype_proj_heps_pt02'].to_dict()
    full_adata.obs['celltype_proj_heps_pt02'] = full_adata.obs.index.map(hepatocyte_mapping)
    
    # fillna BEFORE the str conversion, while un-mapped cells are still true NaN
    full_adata.obs['pred_cell_types'] = full_adata.obs['pred_cell_types'].astype(str)
    full_adata.obs['celltype_proj_heps_pt02'] = full_adata.obs['celltype_proj_heps_pt02'].fillna(
        full_adata.obs['pred_cell_types']
    )
    
    full_adata.obs['celltype_proj_heps_pt02'] = full_adata.obs['celltype_proj_heps_pt02'].astype(str).astype("category")

    # ================= TROUBLESHOOTING: post-merge diagnostics =================
    pre_categories = set(full_adata.obs['pred_cell_types'].unique())
    post_categories = set(full_adata.obs['celltype_proj_heps_pt02'].unique())
    dropped = pre_categories - post_categories
    print(f"[{sample_nm}] celltype_proj_heps_pt02 value counts (after merge):")
    print(full_adata.obs['celltype_proj_heps_pt02'].value_counts())
    if dropped:
        print(
            f"[{sample_nm}] WARNING: these pred_cell_types categories are missing "
            f"from celltype_proj_heps_pt02 after the merge: {sorted(dropped)}"
        )
        for missing_ct in sorted(dropped):
            n_missing = (full_adata.obs['pred_cell_types'] == missing_ct).sum()
            print(f"[{sample_nm}]   '{missing_ct}': {n_missing} cells in pred_cell_types before merge")
    else:
        print(f"[{sample_nm}] OK: all pred_cell_types categories are present in celltype_proj_heps_pt02")
    # ============================================================================

    # --- neighborhood enrichment (co-occurrence intentionally not run here) ---
    adata_sub = full_adata.copy()
    # a plain copy with unused categories dropped avoids squidpy's
    # "N color bins but M colors" mismatch when plotting
    adata_sub.obs[cluster_key] = (
        adata_sub.obs['celltype_proj_heps_pt02'].astype(str).astype('category')
    )

    print(f"[{sample_nm}] running spatial_neighbors / nhood_enrichment...", datetime.datetime.now())
    sq.gr.spatial_neighbors(adata_sub)
    sq.gr.nhood_enrichment(adata_sub, cluster_key=cluster_key, seed=0, show_progress_bar=False)
    print(f"[{sample_nm}] done.", datetime.datetime.now())

    # full enrichment matrix (default RdBu_r colormap)
    sq.pl.nhood_enrichment(adata_sub, cluster_key=cluster_key, seed=0,)
    plt.savefig(f"{odir}enrich_mat_sub_{sample_nm}.pdf", bbox_inches='tight')
    plt.close()

    sq.pl.nhood_enrichment(adata_sub, cluster_key=cluster_key, annotate=True)
    plt.savefig(f"{odir}enrich_mat_sub_anno_{sample_nm}.pdf", bbox_inches='tight')
    plt.close()

    # write out the neighbor-graph object for this sample
    out_path = f"{odir}adata_zones_broad_celltypes_zones_highsen_pt02_neighbor_{sample_nm}.h5ad"
    adata_sub.write_h5ad(out_path)
    print(f"[{sample_nm}] wrote {out_path}")

    # --- zoomed-in heatmap: just the hepatocyte subtype rows, sen.high on top ---
    ct_sort = sorted(adata_sub.obs[cluster_key].unique().tolist())
    enrichment_df = pd.DataFrame(adata_sub.uns[f'{cluster_key}_nhood_enrichment']['zscore'])

    # hardcoded row/column positions (kept as in the original notebook)
    rows_to_plot = enrichment_df.iloc[4:9]
    rows_to_plot = rows_to_plot.reindex([5, 4, 6, 7, 8])

    col_indices = list(range(enrichment_df.shape[1]))
    col_indices[4], col_indices[5] = col_indices[5], col_indices[4]
    rows_to_plot = rows_to_plot.iloc[:, col_indices]

    rows_to_plot_values = rows_to_plot.values
    xticklabels = [ct_sort[i] for i in col_indices]

    for annotate, suffix in [(True, "anno"), (False, "plain")]:
        plt.figure(figsize=(8, 3))
        sns.heatmap(
            rows_to_plot_values,
            cmap='coolwarm',
            fmt='.2f',
            annot=annotate,
            xticklabels=xticklabels,
            yticklabels=heps_labels,
            cbar_kws={'label': 'Enrichment Score'},
        )
        plt.title("Heatmap of Hepatocytes.Senescent.High from Neighborhood Enrichment")
        plt.xlabel("Neighborhoods")
        plt.ylabel("Rows")
        plt.xticks(rotation=90)
        plt.savefig(f"{odir}enrich_mat_sub_only_high_sen_all_heps_{suffix}_{sample_nm}.pdf", bbox_inches='tight')
        plt.close()

    return adata_sub

## Run the pipeline for every sample

In [15]:
adata_sub_by_sample = {}
for sample_nm in sample_nm_list:
    adata_sub_by_sample[sample_nm] = process_sample(sample_nm)


[HL170058_filt] --- troubleshooting celltype_proj_heps_pt02 merge ---
[HL170058_filt] pred_cell_types value counts (full_adata, before merge):
pred_cell_types
Hepatocytes      190195
Myeloid           27802
Endothelial       25542
HSC               25412
Cholangiocyte     18483
B                 15893
T_NK              12404
Name: count, dtype: int64
[HL170058_filt] adata_heps n_obs: 190195 | full_adata n_obs: 315731
[HL170058_filt] barcode overlap (adata_heps ∩ full_adata): 190195
[HL170058_filt] duplicate barcodes -> adata_heps: 0 | full_adata: 0
[HL170058_filt] celltype_proj_heps_pt02 value counts (after merge):
celltype_proj_heps_pt02
Hepatocytes.Zone3             76425
Hepatocytes.Zone2             40529
Hepatocytes.Zone1             37666
Hepatocytes.Senescent         34304
Myeloid                       27802
Endothelial                   25542
HSC                           25412
Cholangiocyte                 18483
B                             15893
T_NK                         